# Polar-Inspired KV-Cache Compression (PyTorch)

This notebook is **self-contained**: it includes all models/quantizers and runs the full experiment sweep, writing outputs to `project/results/`.

In [ ]:
import math
import csv
import time
from dataclasses import dataclass
from pathlib import Path

import torch

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None


def get_device() -> str:
    return "cuda" if torch.cuda.is_available() else "cpu"


DEVICE = get_device()
print("device:", DEVICE)
print("torch:", torch.__version__)
if plt is not None:
    print("matplotlib:", plt.matplotlib.__version__)

In [ ]:
# -----------------
# Utils: quantization
# -----------------

def uniform_quantize(
    x: torch.Tensor,
    bits: int,
    min_val: torch.Tensor | None = None,
    max_val: torch.Tensor | None = None,
):
    if bits <= 0:
        raise ValueError("bits must be positive")

    qmin = 0
    qmax = (1 << bits) - 1

    if min_val is None:
        min_val = x.amin(dim=-1, keepdim=True)
    if max_val is None:
        max_val = x.amax(dim=-1, keepdim=True)

    eps = torch.finfo(x.dtype).eps
    scale = (max_val - min_val).clamp_min(eps) / float(qmax - qmin)
    q = torch.round((x - min_val) / scale).clamp(qmin, qmax).to(torch.int32)
    return q, scale, min_val


def uniform_dequantize(q: torch.Tensor, scale: torch.Tensor, min_val: torch.Tensor):
    return q.to(scale.dtype) * scale + min_val


def seed_everything(seed: int) -> None:
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
# -------------
# Utils: data
# -------------

@dataclass
class KVCacheBatch:
    q: torch.Tensor
    k: torch.Tensor
    v: torch.Tensor


def generate_synthetic_kv(num_tokens: int, d_model: int, device: str = "cpu") -> KVCacheBatch:
    k = torch.randn(num_tokens, d_model, device=device)
    v = torch.randn(num_tokens, d_model, device=device)
    q = torch.randn(1, d_model, device=device)
    return KVCacheBatch(q=q, k=k, v=v)

In [ ]:
# -----------------
# Model: attention
# -----------------

def scaled_dot_product_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor):
    d = q.shape[-1]
    scores = (q @ k.transpose(-1, -2)) / math.sqrt(d)
    probs = torch.softmax(scores, dim=-1)
    output = probs @ v
    return output, probs

In [ ]:
# -----------------
# Eval: metrics
# -----------------

def reconstruction_mse(x: torch.Tensor, y: torch.Tensor) -> float:
    return torch.mean((x - y) ** 2).item()


def cosine_similarity_mean(x: torch.Tensor, y: torch.Tensor, eps: float = 1e-8) -> float:
    x_n = x / x.norm(dim=-1, keepdim=True).clamp_min(eps)
    y_n = y / y.norm(dim=-1, keepdim=True).clamp_min(eps)
    return (x_n * y_n).sum(dim=-1).mean().item()


def norm_preservation_error(x: torch.Tensor, y: torch.Tensor, eps: float = 1e-8) -> float:
    nx = x.norm(dim=-1).clamp_min(eps)
    ny = y.norm(dim=-1)
    return torch.mean(torch.abs(nx - ny) / nx).item()


def kl_divergence(p: torch.Tensor, q: torch.Tensor, eps: float = 1e-8) -> float:
    p = p.clamp_min(eps)
    q = q.clamp_min(eps)
    return torch.mean(torch.sum(p * (torch.log(p) - torch.log(q)), dim=-1)).item()

In [ ]:
# -----------------
# Quantizer: HS3
# -----------------

@dataclass
class HS3Encoded:
    r: torch.Tensor
    phi_q: torch.Tensor
    theta_q: torch.Tensor
    orig_dim: int


class HS3Quantizer:
    def __init__(self, eps: float = 1e-8):
        self.eps = eps

    def _pad(self, x: torch.Tensor):
        d = x.shape[-1]
        pad = (3 - (d % 3)) % 3
        if pad:
            x = torch.nn.functional.pad(x, (0, pad))
        return x, d

    def encode(self, x: torch.Tensor, bits: int) -> HS3Encoded:
        levels = 1 << bits
        x_pad, orig_dim = self._pad(x)
        blocks = x_pad.reshape(*x_pad.shape[:-1], -1, 3)
        bx, by, bz = blocks.unbind(dim=-1)

        r = torch.sqrt((blocks**2).sum(dim=-1) + self.eps)
        theta = torch.acos((bz / r).clamp(-1.0, 1.0))
        phi = torch.atan2(by, bx)
        phi = torch.where(phi < 0, phi + 2 * torch.pi, phi)

        phi_q = torch.round(phi / (2 * torch.pi) * (levels - 1)).clamp(0, levels - 1).to(torch.int16)
        theta_q = torch.round(theta / torch.pi * (levels - 1)).clamp(0, levels - 1).to(torch.int16)
        return HS3Encoded(r=r, phi_q=phi_q, theta_q=theta_q, orig_dim=orig_dim)

    def decode(self, encoded: HS3Encoded, bits: int) -> torch.Tensor:
        levels = 1 << bits
        phi = encoded.phi_q.to(encoded.r.dtype) / (levels - 1) * (2 * torch.pi)
        theta = encoded.theta_q.to(encoded.r.dtype) / (levels - 1) * torch.pi
        r = encoded.r

        sin_theta = torch.sin(theta)
        x = r * sin_theta * torch.cos(phi)
        y = r * sin_theta * torch.sin(phi)
        z = r * torch.cos(theta)

        out = torch.stack([x, y, z], dim=-1).reshape(*encoded.r.shape[:-1], -1)
        return out[..., : encoded.orig_dim]

    @staticmethod
    def bits_per_dim(bits: int) -> float:
        return (32 + 2 * bits) / 3.0

In [ ]:
# -----------------
# Quantizer: CP
# -----------------

@dataclass
class CPEncoded:
    mag: torch.Tensor
    phase_q: torch.Tensor
    mag_q: torch.Tensor | None
    mag_scale: torch.Tensor | None
    mag_zero: torch.Tensor | None
    orig_dim: int


class CPQuantizer:
    def __init__(self, quantize_magnitude: bool = False):
        self.quantize_magnitude = quantize_magnitude

    def _pad(self, x: torch.Tensor):
        d = x.shape[-1]
        pad = d % 2
        if pad:
            x = torch.nn.functional.pad(x, (0, 1))
        return x, d

    def encode(self, x: torch.Tensor, bits: int) -> CPEncoded:
        levels = 1 << bits
        x_pad, orig_dim = self._pad(x)
        pairs = x_pad.reshape(*x_pad.shape[:-1], -1, 2)
        real = pairs[..., 0]
        imag = pairs[..., 1]

        mag = torch.sqrt(real**2 + imag**2 + 1e-8)
        phase = torch.atan2(imag, real)
        phase = torch.where(phase < 0, phase + 2 * torch.pi, phase)
        phase_q = torch.round(phase / (2 * torch.pi) * (levels - 1)).clamp(0, levels - 1).to(torch.int16)

        if self.quantize_magnitude:
            mag_q, mag_scale, mag_zero = uniform_quantize(mag, bits)
            mag_store = None
        else:
            mag_q, mag_scale, mag_zero = None, None, None
            mag_store = mag

        return CPEncoded(
            mag=mag_store,
            phase_q=phase_q,
            mag_q=mag_q,
            mag_scale=mag_scale,
            mag_zero=mag_zero,
            orig_dim=orig_dim,
        )

    def decode(self, encoded: CPEncoded, bits: int) -> torch.Tensor:
        levels = 1 << bits
        phase = encoded.phase_q.to(torch.float32) / (levels - 1) * (2 * torch.pi)

        if encoded.mag_q is None:
            mag = encoded.mag
        else:
            mag = uniform_dequantize(encoded.mag_q, encoded.mag_scale, encoded.mag_zero)

        real = mag * torch.cos(phase)
        imag = mag * torch.sin(phase)
        out = torch.stack([real, imag], dim=-1).reshape(*phase.shape[:-1], -1)
        return out[..., : encoded.orig_dim]

    def bits_per_dim(self, bits: int) -> float:
        mag_bits = bits if self.quantize_magnitude else 32
        return (bits + mag_bits) / 2.0

In [ ]:
# -----------------
# Quantizer: RPU
# -----------------

@dataclass
class RPUEncoded:
    q: torch.Tensor
    scale: torch.Tensor
    zero: torch.Tensor
    norms: torch.Tensor


class RPUQuantizer:
    def __init__(self, d_model: int, seed: int, normalize: bool = True):
        self.normalize = normalize
        g = torch.Generator(device="cpu")
        g.manual_seed(seed)
        a = torch.randn(d_model, d_model, generator=g)
        q, _ = torch.linalg.qr(a, mode="reduced")
        self.Q = q

    def encode(self, x: torch.Tensor, bits: int) -> RPUEncoded:
        Q = self.Q.to(x.device, x.dtype)
        rot = x @ Q.T
        if self.normalize:
            norms = rot.norm(dim=-1, keepdim=True).clamp_min(1e-8)
            rot_to_q = rot / norms
        else:
            norms = torch.ones_like(rot[..., :1])
            rot_to_q = rot

        q, scale, zero = uniform_quantize(rot_to_q, bits)
        return RPUEncoded(q=q, scale=scale, zero=zero, norms=norms)

    def decode(self, encoded: RPUEncoded, bits: int) -> torch.Tensor:
        del bits
        rot = uniform_dequantize(encoded.q, encoded.scale, encoded.zero)
        if self.normalize:
            rot = rot * encoded.norms
        Q = self.Q.to(rot.device, rot.dtype)
        return rot @ Q

    @staticmethod
    def bits_per_dim(bits: int) -> float:
        return float(bits)

In [ ]:
# -----------------
# Baseline KV-cache compression
# -----------------
# Baseline: direct per-token uniform quantization of K/V (no transforms).

@dataclass
class BaselineEncoded:
    q: torch.Tensor
    scale: torch.Tensor
    zero: torch.Tensor


class BaselineKVQuantizer:
    def encode(self, x: torch.Tensor, bits: int) -> BaselineEncoded:
        q, scale, zero = uniform_quantize(x, bits)
        return BaselineEncoded(q=q, scale=scale, zero=zero)

    def decode(self, encoded: BaselineEncoded, bits: int) -> torch.Tensor:
        del bits
        return uniform_dequantize(encoded.q, encoded.scale, encoded.zero)

    @staticmethod
    def bits_per_dim(bits: int) -> float:
        return float(bits)

In [ ]:
# -----------------
# Runner: experiments
# -----------------

@dataclass
class EvalConfig:
    bits_list: list[int]
    seeds: list[int]
    dims: list[int]
    token_choices: list[int]
    output_dir: str = "project/results"
    device: str = DEVICE


def _compress_pair(quantizer, k: torch.Tensor, v: torch.Tensor, bits: int):
    k_enc = quantizer.encode(k, bits)
    v_enc = quantizer.encode(v, bits)
    k_hat = quantizer.decode(k_enc, bits)
    v_hat = quantizer.decode(v_enc, bits)
    return k_hat, v_hat


def _jl_distortion(x: torch.Tensor, y: torch.Tensor, num_pairs: int = 64) -> float:
    n = x.shape[0]
    idx_a = torch.randint(0, n, (num_pairs,), device=x.device)
    idx_b = torch.randint(0, n, (num_pairs,), device=x.device)
    dx = (x[idx_a] - x[idx_b]).norm(dim=-1)
    dy = (y[idx_a] - y[idx_b]).norm(dim=-1)
    return (torch.abs(dy - dx) / dx.clamp_min(1e-8)).mean().item()


def _plot_metric(rows, out_dir: Path, metric: str, filename: str, ylabel: str):
    methods = sorted({r["method"] for r in rows})
    bits_list = sorted({r["bits"] for r in rows})

    plt.figure(figsize=(8, 5))
    for m in methods:
        ys = []
        for b in bits_list:
            vals = [r[metric] for r in rows if r["method"] == m and r["bits"] == b]
            ys.append(sum(vals) / len(vals))
        plt.plot(bits_list, ys, marker="o", label=m)

    plt.xlabel("Bit-width")
    plt.ylabel(ylabel)
    plt.title(f"{ylabel} vs bit-width")
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dir / filename, dpi=140)
    plt.close()


def summarize(rows: list[dict]) -> str:
    def best_for(bit_filter):
        subset = [r for r in rows if bit_filter(r["bits"])]
        grouped: dict[str, list[float]] = {}
        for r in subset:
            grouped.setdefault(r["method"], []).append(r["attn_out_mse"])
        means = {k: sum(v) / len(v) for k, v in grouped.items()}
        best = min(means, key=means.get)
        return best, means[best]

    low_best, low_val = best_for(lambda b: b <= 4)
    high_best, high_val = best_for(lambda b: b >= 6)

    return (
        "\n=== Summary report ===\n"
        f"Best method (low-bit, <=4): {low_best} (mean attn MSE={low_val:.6f})\n"
        f"Best method (high-bit, >=6): {high_best} (mean attn MSE={high_val:.6f})\n"
        "RPU JL-style check: see jl_distortion column (lower is better).\n"
    )


def run_experiments(cfg: EvalConfig):
    out_dir = Path(cfg.output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    rows: list[dict] = []
    methods = [
        ("Baseline-uniform", lambda d, s: BaselineKVQuantizer(), lambda q, b: q.bits_per_dim(b)),
        ("HS3", lambda d, s: HS3Quantizer(), lambda q, b: q.bits_per_dim(b)),
        ("CP-phase", lambda d, s: CPQuantizer(quantize_magnitude=False), lambda q, b: q.bits_per_dim(b)),
        ("CP-phase+mag", lambda d, s: CPQuantizer(quantize_magnitude=True), lambda q, b: q.bits_per_dim(b)),
        ("RPU-norm", lambda d, s: RPUQuantizer(d_model=d, seed=s + 13, normalize=True), lambda q, b: q.bits_per_dim(b)),
        ("RPU-no-norm", lambda d, s: RPUQuantizer(d_model=d, seed=s + 13, normalize=False), lambda q, b: q.bits_per_dim(b)),
    ]

    for seed in cfg.seeds:
        seed_everything(seed)
        for d_model in cfg.dims:
            for num_tokens in cfg.token_choices:
                batch = generate_synthetic_kv(num_tokens=num_tokens, d_model=d_model, device=cfg.device)
                ref_out, ref_probs = scaled_dot_product_attention(batch.q, batch.k, batch.v)

                for bits in cfg.bits_list:
                    for method_name, ctor, bpd_fn in methods:
                        quantizer = ctor(d_model, seed)
                        t0 = time.perf_counter()
                        k_hat, v_hat = _compress_pair(quantizer, batch.k, batch.v, bits)
                        out_hat, probs_hat = scaled_dot_product_attention(batch.q, k_hat, v_hat)
                        dt = time.perf_counter() - t0

                        rows.append(
                            {
                                "seed": seed,
                                "d_model": d_model,
                                "num_tokens": num_tokens,
                                "bits": bits,
                                "method": method_name,
                                "k_mse": reconstruction_mse(batch.k, k_hat),
                                "v_mse": reconstruction_mse(batch.v, v_hat),
                                "k_cos": cosine_similarity_mean(batch.k, k_hat),
                                "v_cos": cosine_similarity_mean(batch.v, v_hat),
                                "k_norm_err": norm_preservation_error(batch.k, k_hat),
                                "v_norm_err": norm_preservation_error(batch.v, v_hat),
                                "attn_out_mse": reconstruction_mse(ref_out, out_hat),
                                "attn_out_cos": cosine_similarity_mean(ref_out, out_hat),
                                "attn_kl": kl_divergence(ref_probs, probs_hat),
                                "jl_distortion": _jl_distortion(batch.k, k_hat),
                                "bits_per_dim": bpd_fn(quantizer, bits),
                                "compression_ratio": 32.0 / bpd_fn(quantizer, bits),
                                "runtime_ms": 1000.0 * dt,
                            }
                        )

    csv_path = out_dir / "results.csv"
    with csv_path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

    if plt is not None:
        _plot_metric(rows, out_dir, metric="k_mse", filename="mse_vs_bits.png", ylabel="Key reconstruction MSE")
        _plot_metric(rows, out_dir, metric="attn_out_mse", filename="attention_error_vs_bits.png", ylabel="Attention output MSE")
        _plot_metric(rows, out_dir, metric="attn_out_cos", filename="attention_cosine_vs_bits.png", ylabel="Attention output cosine")
    else:
        print("matplotlib not installed; skipping plot generation.")

    summary = summarize(rows)
    print(summary)
    return rows, summary

In [ ]:
# -----------------
# Run: full sweep
# -----------------

cfg = EvalConfig(
    bits_list=[2, 3, 4, 6, 8],
    seeds=[0, 1, 2, 3, 4],
    dims=[64, 128, 256],
    token_choices=[32, 64, 128],
)

rows, summary = run_experiments(cfg)
len(rows)

In [ ]:
# Quick peek at results and generated plots

out_dir = Path(cfg.output_dir)
print("wrote:", out_dir / "results.csv")

if plt is not None:
    from IPython.display import display

    for name in ["mse_vs_bits.png", "attention_error_vs_bits.png", "attention_cosine_vs_bits.png"]:
        p = out_dir / name
        if p.exists():
            img = plt.imread(p)
            plt.figure(figsize=(8, 5))
            plt.imshow(img)
            plt.axis("off")
            plt.title(name)
            plt.show()
        else:
            print("missing:", p)
else:
    print("matplotlib not installed; plots were skipped")

In [ ]:
# -----------------
# Accuracy comparison (Baseline vs HS3/CP/RPU)
# -----------------

from collections import defaultdict


def _mean(xs):
    return sum(xs) / max(1, len(xs))


def summarize_accuracy(rows: list[dict], metric: str = "attn_out_mse"):
    # aggregate mean(metric) over seeds/dims/tokens for each (bits, method)
    agg = defaultdict(list)
    for r in rows:
        agg[(r["bits"], r["method"])].append(float(r[metric]))

    bits_list = sorted({r["bits"] for r in rows})
    methods = sorted({r["method"] for r in rows})

    table = []
    for b in bits_list:
        row = {"bits": b}
        for m in methods:
            row[m] = _mean(agg[(b, m)])
        table.append(row)

    # print a simple aligned table (no pandas dependency)
    col_names = ["bits"] + methods
    widths = {c: max(len(c), 12) for c in col_names}
    for r in table:
        for c in col_names:
            widths[c] = max(widths[c], len(f"{r[c]:.6g}" if c != "bits" else str(r[c])))

    def fmt_row(r):
        out = []
        for c in col_names:
            if c == "bits":
                out.append(str(r[c]).rjust(widths[c]))
            else:
                out.append(f"{r[c]:.6g}".rjust(widths[c]))
        return "  ".join(out)

    header = "  ".join(c.rjust(widths[c]) for c in col_names)
    sep = "  ".join("-" * widths[c] for c in col_names)
    print(f"\nMean {metric} (lower is better)\n")
    print(header)
    print(sep)
    for r in table:
        print(fmt_row(r))

    # compute relative improvement vs baseline for each method and bit-width
    baseline_name = "Baseline-uniform"
    if baseline_name in methods:
        print("\nΔ vs baseline (negative is better, baseline subtracted)\n")
        for b in bits_list:
            base = _mean(agg[(b, baseline_name)])
            deltas = []
            for m in methods:
                if m == baseline_name:
                    continue
                deltas.append((m, _mean(agg[(b, m)]) - base))
            deltas.sort(key=lambda x: x[1])
            pretty = ", ".join(f"{m}: {d:+.3g}" for m, d in deltas)
            print(f"bits={b}: {pretty}")


summarize_accuracy(rows, metric="attn_out_mse")
summarize_accuracy(rows, metric="attn_out_cos")

## Transformer KV-cache experiment

This experiment uses a small decoder-only Transformer with a KV cache. We compare KV-cache compression methods by:

- **Accuracy**: agreement with full-precision (top-1 match rate) and mean KL(full || compressed)
- **Speed**: average ms/token over a short autoregressive run
- **Results**: table broken down by method and bit-width

In [ ]:
import torch.nn as nn


def _sync_if_cuda(device: str):
    if device.startswith("cuda") and torch.cuda.is_available():
        torch.cuda.synchronize()


class ToyDecoder(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, seed: int = 0):
        super().__init__()
        g = torch.Generator(device="cpu")
        g.manual_seed(seed)

        self.vocab_size = vocab_size
        self.d_model = d_model

        self.tok = nn.Embedding(vocab_size, d_model)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)
        self.ln = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # deterministic init
        with torch.no_grad():
            for p in self.parameters():
                p.copy_(torch.randn_like(p, generator=g) * 0.02)

    def step(self, token_id: torch.Tensor, k_cache: torch.Tensor | None, v_cache: torch.Tensor | None):
        """One autoregressive step.

        token_id: [B]
        k_cache/v_cache: [T, B, D] or None
        returns: logits [B, V], new_k_cache, new_v_cache
        """
        x = self.tok(token_id)  # [B, D]
        x = self.ln(x)

        q = self.q_proj(x).unsqueeze(0)  # [1, B, D]
        k_new = self.k_proj(x).unsqueeze(0)
        v_new = self.v_proj(x).unsqueeze(0)

        if k_cache is None:
            k_cat = k_new
            v_cat = v_new
        else:
            k_cat = torch.cat([k_cache, k_new], dim=0)
            v_cat = torch.cat([v_cache, v_new], dim=0)

        # single-head attention with caching
        # q: [1,B,D] ; k_cat/v_cat: [T,B,D]
        attn_scores = (q * k_cat).sum(dim=-1) / math.sqrt(self.d_model)  # [T,B]
        attn_probs = torch.softmax(attn_scores, dim=0)  # softmax over time
        ctx = (attn_probs.unsqueeze(-1) * v_cat).sum(dim=0)  # [B,D]

        y = self.o_proj(ctx)
        logits = self.lm_head(y)  # [B,V]
        return logits, k_cat, v_cat


def _compress_cache(method_name: str, bits: int, d_model: int, seed: int, k: torch.Tensor, v: torch.Tensor):
    """Compress+decompress full cache tensors k/v shaped [T,B,D]."""
    if method_name == "Baseline-uniform":
        q = BaselineKVQuantizer()
    elif method_name == "HS3":
        q = HS3Quantizer()
    elif method_name == "CP-phase":
        q = CPQuantizer(quantize_magnitude=False)
    elif method_name == "CP-phase+mag":
        q = CPQuantizer(quantize_magnitude=True)
    elif method_name == "RPU-norm":
        q = RPUQuantizer(d_model=d_model, seed=seed + 13, normalize=True)
    elif method_name == "RPU-no-norm":
        q = RPUQuantizer(d_model=d_model, seed=seed + 13, normalize=False)
    else:
        raise ValueError(f"unknown method: {method_name}")

    # flatten [T,B,D] -> [T*B, D] so quantizers operate per token vector
    tb = k.shape[0] * k.shape[1]
    k2 = k.reshape(tb, d_model)
    v2 = v.reshape(tb, d_model)

    k_hat = q.decode(q.encode(k2, bits), bits).reshape_as(k)
    v_hat = q.decode(q.encode(v2, bits), bits).reshape_as(v)
    return k_hat, v_hat


def run_transformer_kvcache_experiment(
    device: str = DEVICE,
    seed: int = 0,
    vocab_size: int = 2048,
    d_model: int = 256,
    batch_size: int = 8,
    prompt_len: int = 32,
    gen_len: int = 32,
    bits_list: list[int] = [2, 3, 4, 6, 8],
):
    torch.manual_seed(seed)

    model = ToyDecoder(vocab_size=vocab_size, d_model=d_model, seed=seed).to(device)
    model.eval()

    # fixed prompt tokens
    g = torch.Generator(device="cpu")
    g.manual_seed(seed + 999)
    prompt = torch.randint(0, vocab_size, (prompt_len, batch_size), generator=g, device=device)

    methods = [
        "Baseline-uniform",
        "HS3",
        "CP-phase",
        "CP-phase+mag",
        "RPU-norm",
        "RPU-no-norm",
    ]

    # build full-precision cache from prompt
    with torch.no_grad():
        k_fp, v_fp = None, None
        for t in range(prompt_len):
            _logits, k_fp, v_fp = model.step(prompt[t], k_fp, v_fp)

    # reference run (no compression) for the generation window
    with torch.no_grad():
        tok = prompt[-1]
        k_ref, v_ref = k_fp, v_fp
        ref_logits = []
        for _ in range(gen_len):
            lg, k_ref, v_ref = model.step(tok, k_ref, v_ref)
            ref_logits.append(lg)
            tok = torch.argmax(lg, dim=-1)
        ref_logits = torch.stack(ref_logits, dim=0)  # [gen,B,V]

    results = []
    for bits in bits_list:
        for m in methods:
            with torch.no_grad():
                tok = prompt[-1]
                k_c, v_c = k_fp, v_fp

                _sync_if_cuda(device)
                t0 = time.perf_counter()

                top1_matches = 0
                total = 0
                kl_sum = 0.0

                for step_idx in range(gen_len):
                    # compress the cache before using it
                    k_use, v_use = _compress_cache(m, bits, d_model, seed, k_c, v_c)
                    lg, k_c, v_c = model.step(tok, k_use, v_use)

                    ref_lg = ref_logits[step_idx]
                    top1_matches += (torch.argmax(lg, dim=-1) == torch.argmax(ref_lg, dim=-1)).sum().item()
                    total += tok.shape[0]

                    p = torch.softmax(ref_lg, dim=-1).clamp_min(1e-8)
                    q = torch.softmax(lg, dim=-1).clamp_min(1e-8)
                    kl_sum += torch.mean(torch.sum(p * (torch.log(p) - torch.log(q)), dim=-1)).item()

                    tok = torch.argmax(lg, dim=-1)

                _sync_if_cuda(device)
                dt = time.perf_counter() - t0

            results.append(
                {
                    "bits": bits,
                    "method": m,
                    "top1_match": top1_matches / max(1, total),
                    "kl_full_vs_comp": kl_sum / float(gen_len),
                    "ms_per_token": (1000.0 * dt) / float(gen_len * batch_size),
                }
            )

    return results


tx_results = run_transformer_kvcache_experiment()
print("rows:", len(tx_results))

In [ ]:
# Results: print and (optionally) plot

from collections import defaultdict


def print_tx_table(rows: list[dict], sort_methods: bool = True):
    bits_list = sorted({r["bits"] for r in rows})
    methods = sorted({r["method"] for r in rows}) if sort_methods else list({r["method"] for r in rows})

    # organize
    by = {(r["bits"], r["method"]): r for r in rows}

    def print_metric(metric: str, fmt: str):
        print(f"\nTransformer KV-cache: {metric}\n")
        header = ["bits"] + methods
        widths = {c: max(len(c), 12) for c in header}
        for b in bits_list:
            for m in methods:
                widths[m] = max(widths[m], len(fmt.format(by[(b, m)][metric])))

        print("  ".join(c.rjust(widths[c]) for c in header))
        print("  ".join("-" * widths[c] for c in header))
        for b in bits_list:
            line = [str(b).rjust(widths["bits"])]
            for m in methods:
                line.append(fmt.format(by[(b, m)][metric]).rjust(widths[m]))
            print("  ".join(line))

    print_metric("top1_match", "{:.4f}")
    print_metric("kl_full_vs_comp", "{:.4g}")
    print_metric("ms_per_token", "{:.3f}")


print_tx_table(tx_results)

if plt is not None:
    # Plot KL vs bits (lower is better)
    methods = sorted({r["method"] for r in tx_results})
    bits_list = sorted({r["bits"] for r in tx_results})
    for metric, ylabel in [
        ("kl_full_vs_comp", "KL(full || compressed)"),
        ("ms_per_token", "ms / token"),
    ]:
        plt.figure(figsize=(8, 5))
        for m in methods:
            ys = []
            for b in bits_list:
                vals = [r[metric] for r in tx_results if r["method"] == m and r["bits"] == b]
                ys.append(sum(vals) / len(vals))
            plt.plot(bits_list, ys, marker="o", label=m)
        plt.xlabel("Bit-width")
        plt.ylabel(ylabel)
        plt.title(f"Transformer KV-cache: {ylabel} vs bit-width")
        plt.grid(True, alpha=0.25)
        plt.legend()
        plt.tight_layout()
        plt.show()